# DNABERT-2 Fine-tuning: Tumor vs Normal Read Classification

**Project:** Genomic-RawSeq-Analyzer — Semester 2, Issue #3  
**Goal:** Replace 1D-CNN + integer encoding with DNABERT-2 BPE tokenization  
**Model:** `zhihan1996/DNABERT-2-117M`  
**Dataset:** Breast cancer WXS cohort (ENA: ERR166302–ERR166337)  
**Baseline:** 1D-CNN read-level AUC = 0.615

---
### Architecture change
| Component | Baseline (1D-CNN) | This notebook (DNABERT-2) |
|-----------|-------------------|---------------------------|
| Encoding | Integer (A=1,C=2,G=3,T=4) | BPE tokenization |
| Read length | 80 bp | up to 512 bp |
| Encoder | Embedding + 2× Conv1D | 12-layer BERT transformer |
| Head | Dense(64) + Dense(1) | Linear(768 → 1) on [CLS] |
| Parameters | ~50 K | ~117 M |

> **Requirements:** Colab Pro+ with A100 GPU (40 GB VRAM recommended)

## 1 · GPU & environment check

In [ ]:
import subprocess, sys

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — switch runtime to A100!')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Install dependencies

In [ ]:
!pip install -q transformers>=4.40 accelerate>=0.28 sentencepiece>=0.1.99 einops>=0.7
!pip install -q scikit-learn matplotlib seaborn biopython
print('All dependencies installed.')

## 3 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
# Folder in your Drive where the batch_*.npz files live
BATCH_DIR = '/content/drive/MyDrive/492/results/batches'

# Where checkpoints and outputs will be saved
OUTPUT_DIR = '/content/drive/MyDrive/492/results/dnabert2'
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
batch_files = sorted([f for f in os.listdir(BATCH_DIR) if f.endswith('.npz')])
print(f'Found {len(batch_files)} batch files in {BATCH_DIR}:')
for f in batch_files:
    print(' ', f)

## 4 · Imports & logging

In [ ]:
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, roc_curve

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 5 · Configuration

In [ ]:
# ─── Hyperparameters ──────────────────────────────────────────────────────────
MODEL_NAME    = 'zhihan1996/DNABERT-2-117M'

# Token budget for DNABERT-2 (BPE tokens, not base-pairs).
# Existing batches contain 80 bp reads → ~20-40 BPE tokens, well within limit.
# Set to 512 if you re-process FASTQ files with longer reads.
MAX_LENGTH    = 128

BATCH_SIZE    = 32    # reduce to 16 if OOM on smaller GPUs
EPOCHS        = 5
LR            = 2e-5  # AdamW peak lr (typical BERT fine-tuning range: 1e-5 – 5e-5)
WARMUP_STEPS  = 200
DROPOUT       = 0.1
TEST_SIZE     = 0.2
RANDOM_SEED   = 42
USE_AMP       = True  # automatic mixed precision (bf16 on A100)
FREEZE_LAYERS = 0     # freeze first N encoder layers (0 = full fine-tune)

print('Config OK.')

## 6 · Load & decode existing batch files

The batch files store **integer-encoded** reads produced by the Semester 1 pipeline.  
We decode them back to raw DNA strings for DNABERT-2's BPE tokenizer.

> **Optional 512 bp path:** If you want longer reads (512 bp), see the cell at the  
> bottom of this section to re-download FASTQ files and extract longer reads.

In [ ]:
# Reverse encoding from data_loader.py
_INT_TO_BASE = {0: '', 1: 'A', 2: 'C', 3: 'G', 4: 'T', 5: 'N'}

def decode_batch(X_int: np.ndarray) -> list:
    """Decode integer matrix → list of raw DNA strings (padding stripped)."""
    return [''.join(_INT_TO_BASE[int(b)] for b in row) for row in X_int]


def load_batches(batch_dir: str):
    """Load all batch_*.npz files, decode sequences, return (sequences, y, run_ids)."""
    import glob
    files = sorted(glob.glob(os.path.join(batch_dir, 'batch_*.npz')))
    if not files:
        raise FileNotFoundError(f'No batch files found in {batch_dir}')

    X_parts, y_parts, run_id_parts = [], [], []
    has_run_ids = True

    for f in files:
        with np.load(f, allow_pickle=True) as data:
            X_parts.append(data['X'])
            y_parts.append(data['y'])
            if 'run_ids' in data:
                run_id_parts.append(data['run_ids'])
            else:
                has_run_ids = False
        print(f'  Loaded {os.path.basename(f)}')

    X_int = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    run_ids = np.concatenate(run_id_parts, axis=0) if has_run_ids else None

    sequences = decode_batch(X_int)

    print(f'\nTotal reads : {len(sequences):,}')
    print(f'Normal (0)  : {(y == 0).sum():,}')
    print(f'Tumor  (1)  : {(y == 1).sum():,}')
    if run_ids is not None:
        print(f'Unique runs : {len(np.unique(run_ids))}')
    print(f'\nExample sequence (first 60 bp): {sequences[0][:60]}…')
    return sequences, y.astype(np.float32), run_ids


sequences, y, run_ids = load_batches(BATCH_DIR)

In [ ]:
# ── OPTIONAL: Re-download FASTQ with 512 bp reads ─────────────────────────────
# Skip this cell if you are happy using the existing 80 bp batches above.
# Uncomment and run ONLY if you want to re-process with longer reads.

# import gzip, subprocess
# from Bio import SeqIO
#
# LONG_MAX_LEN = 512   # base-pairs (BPE will produce fewer tokens)
# READS_PER_RUN = 10_000  # reduce if Drive space is tight
#
# NORMAL_RUNS = [f'ERR1663{i:02d}' for i in range(16, 30)]
# TUMOR_RUNS  = [f'ERR1663{i:02d}' for i in list(range(2, 16)) + list(range(30, 38))]
#
# def download_reads(run_id, label, max_len=512, n_reads=10_000):
#     sub = run_id[:6]
#     url = f'http://ftp.sra.ebi.ac.uk/vol1/fastq/{sub}/{run_id}/{run_id}_1.fastq.gz'
#     seqs, labels, ids = [], [], []
#     try:
#         p = subprocess.Popen(['curl', '-s', '-f', url], stdout=subprocess.PIPE)
#         with gzip.open(p.stdout, 'rt') as h:
#             for rec in SeqIO.parse(h, 'fastq'):
#                 seqs.append(str(rec.seq)[:max_len])
#                 labels.append(label)
#                 ids.append(run_id)
#                 if len(seqs) >= n_reads:
#                     break
#         p.terminate()
#     except Exception as e:
#         print(f'  Error {run_id}: {e}')
#     return seqs, labels, ids
#
# sequences, y_list, run_ids_list = [], [], []
# for rid in NORMAL_RUNS:
#     s, l, i = download_reads(rid, 0, LONG_MAX_LEN, READS_PER_RUN)
#     sequences += s; y_list += l; run_ids_list += i
#     print(f'{rid}: {len(s)} reads')
# for rid in TUMOR_RUNS:
#     s, l, i = download_reads(rid, 1, LONG_MAX_LEN, READS_PER_RUN)
#     sequences += s; y_list += l; run_ids_list += i
#     print(f'{rid}: {len(s)} reads')
#
# y = np.array(y_list, dtype=np.float32)
# run_ids = np.array(run_ids_list)
# MAX_LENGTH = 512   # update tokenizer budget
# print(f'Re-downloaded: {len(sequences):,} reads at up to {LONG_MAX_LEN} bp')
print('Optional re-download cell — skipped (using existing 80 bp batches).')

## 7 · Load DNABERT-2 tokenizer

In [ ]:
print(f'Loading tokenizer from {MODEL_NAME} …')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f'Vocab size : {tokenizer.vocab_size}')
print(f'Max model input : {tokenizer.model_max_length}')

# Spot-check tokenization
sample = sequences[0]
enc = tokenizer(sample, max_length=MAX_LENGTH, truncation=True)
print(f'\nExample sequence ({len(sample)} bp):')
print(f'  {sample[:60]}…')
print(f'→ {len(enc["input_ids"])} BPE tokens (max_length={MAX_LENGTH})')

## 8 · Dataset class & train/test split

In [ ]:
class DNADataset(Dataset):
    """
    Tokenizes raw DNA strings with the DNABERT-2 BPE tokenizer.
    """
    def __init__(self, sequences, labels, tokenizer, max_length=128):
        self.sequences = sequences
        self.labels = np.array(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sequences[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.float32),
        }


# ── Stratified 80/20 split — same ratio as the CNN baseline ────────────────────
idx_train, idx_test = train_test_split(
    np.arange(len(sequences)),
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)

seq_train = [sequences[i] for i in idx_train]
seq_test  = [sequences[i] for i in idx_test]
y_train   = y[idx_train]
y_test    = y[idx_test]
run_ids_test = run_ids[idx_test] if run_ids is not None else None

train_ds = DNADataset(seq_train, y_train, tokenizer, MAX_LENGTH)
test_ds  = DNADataset(seq_test,  y_test,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds):,} reads  |  Test: {len(test_ds):,} reads')
print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

## 9 · DNABERT-2 Classifier model

In [ ]:
class DNABERT2Classifier(nn.Module):
    """
    Binary classifier: DNABERT-2 encoder + linear head on [CLS] token.

    Forward:
        DNABERT-2(input_ids, attention_mask)
          → last_hidden_state[:, 0, :]   ([CLS], shape: batch × 768)
          → Dropout
          → Linear(768, 1)
          → Sigmoid
          → cancer probability per read
    """
    def __init__(self, model_name=MODEL_NAME, dropout=DROPOUT, freeze_layers=FREEZE_LAYERS):
        super().__init__()
        print(f'Loading encoder from {model_name} …')
        self.encoder = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        hidden = self.encoder.config.hidden_size  # 768
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

        # Optionally freeze the first `freeze_layers` transformer layers
        if freeze_layers > 0:
            for layer in self.encoder.encoder.layer[:freeze_layers]:
                for p in layer.parameters():
                    p.requires_grad = False
            print(f'Froze first {freeze_layers} encoder layers.')

        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Parameters: {total/1e6:.1f} M total  |  {trainable/1e6:.1f} M trainable')

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token
        return torch.sigmoid(self.head(self.drop(cls))).squeeze(-1)


model = DNABERT2Classifier()
model = model.to(DEVICE)

## 10 · Optimizer, scheduler & AMP setup

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

# Automatic mixed precision: bfloat16 on A100 for speed + stability
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == 'cuda')
criterion = nn.BCELoss()

print(f'Total training steps: {total_steps}  |  Warmup: {WARMUP_STEPS}')
print(f'AMP enabled: {USE_AMP}')

## 11 · Training loop

In [ ]:
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad if training else torch.no_grad
    with ctx():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            labs  = batch['label'].to(DEVICE)

            if training:
                optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == 'cuda'):
                preds = model(ids, masks)
                loss  = criterion(preds, labs)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item() * len(labs)
            correct    += ((preds >= 0.5).float() == labs).sum().item()
            total      += len(labs)
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(labs.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    auc    = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else float('nan')
    return total_loss / total, correct / total, auc, y_true, y_pred


history = {'train_loss': [], 'train_acc': [], 'train_auc': [],
           'val_loss':   [], 'val_acc':   [], 'val_auc':   []}
best_auc = 0.0
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'best_checkpoint')
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f'Fine-tuning DNABERT-2 for {EPOCHS} epochs …\n')

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_auc, _, _       = run_epoch(train_loader, training=True)
    val_loss, val_acc, val_auc, y_true, y_pred = run_epoch(test_loader,  training=False)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['train_auc'].append(tr_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    flag = ''
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_PATH, 'pytorch_model.pt'))
        tokenizer.save_pretrained(CHECKPOINT_PATH)
        model.encoder.config.save_pretrained(CHECKPOINT_PATH)
        flag = ' ← best'

    print(f'Epoch {epoch}/{EPOCHS} | '
          f'train loss={tr_loss:.4f} acc={tr_acc:.4f} auc={tr_auc:.4f} | '
          f'val loss={val_loss:.4f} acc={val_acc:.4f} auc={val_auc:.4f}{flag}')

print(f'\nBest validation AUC: {best_auc:.4f}')

## 12 · Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_x = range(1, EPOCHS + 1)

axes[0].plot(epochs_x, history['train_loss'], 'b-o', label='train')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', label='val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs_x, history['train_acc'], 'b-o', label='train')
axes[1].plot(epochs_x, history['val_acc'],   'r-o', label='val')
axes[1].set_title('Accuracy'); axes[1].legend()

axes[2].plot(epochs_x, history['train_auc'], 'b-o', label='train')
axes[2].plot(epochs_x, history['val_auc'],   'r-o', label='val')
axes[2].axhline(0.615, color='gray', linestyle='--', label='CNN baseline')
axes[2].set_title('AUC'); axes[2].legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'dnabert2_training_history.png'), dpi=150)
plt.show()
print('Training history saved.')

## 13 · Load best checkpoint & final evaluation

In [ ]:
# Load best weights
model.load_state_dict(torch.load(
    os.path.join(CHECKPOINT_PATH, 'pytorch_model.pt'),
    map_location=DEVICE
))
model.eval()

_, _, _, y_true_final, y_pred_final = run_epoch(test_loader, training=False)

final_auc = roc_auc_score(y_true_final, y_pred_final)
print(f'\n=== Final Read-Level Evaluation ===')
print(f'AUC : {final_auc:.4f}  (CNN baseline: 0.615)')
print()
print(classification_report(
    y_true_final, (y_pred_final >= 0.5).astype(int),
    target_names=['Normal', 'Tumor']
))

## 14 · ROC curve — read level

In [ ]:
fpr, tpr, _ = roc_curve(y_true_final, y_pred_final)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'DNABERT-2 (AUC={final_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
plt.axhline(0, color='gray', lw=0.5)

# CNN baseline reference (approximate from Semester 1)
plt.plot([], [], 'g--', label='CNN baseline (AUC≈0.615)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('DNABERT-2 vs CNN — Read-Level ROC')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dnabert2_roc_read_level.png'), dpi=150)
plt.show()

## 15 · Patient-level aggregation (crowd voting)

In [ ]:
if run_ids_test is None:
    print('run_ids not available — skipping patient-level aggregation.')
else:
    patient_df = pd.DataFrame({
        'run_id':  run_ids_test,
        'y_true':  y_true_final,
        'y_pred':  y_pred_final,
    })

    # Aggregate: mean prediction probability per patient
    agg = patient_df.groupby('run_id').agg(
        patient_label=('y_true', lambda x: int(x.mode()[0])),
        patient_prob=('y_pred', 'mean'),
        n_reads=('y_pred', 'count'),
    ).reset_index()

    patient_auc = roc_auc_score(agg['patient_label'], agg['patient_prob'])
    print(f'Patient-level AUC : {patient_auc:.4f}  (n={len(agg)} patients)')
    print(f'Read-level    AUC : {final_auc:.4f}')
    print()
    print(agg.to_string(index=False))

    # ROC
    fpr_p, tpr_p, _ = roc_curve(agg['patient_label'], agg['patient_prob'])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr_p, tpr_p, lw=2, color='darkorange',
             label=f'Patient-level (AUC={patient_auc:.3f})')
    plt.plot(fpr, tpr, lw=2, color='steelblue',
             label=f'Read-level (AUC={final_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('DNABERT-2 — Patient vs Read Level ROC')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dnabert2_patient_roc.png'), dpi=150)
    plt.show()

## 16 · Prediction distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(y_pred_final[y_true_final == 0], bins=50, alpha=0.6, label='Normal', color='steelblue')
plt.hist(y_pred_final[y_true_final == 1], bins=50, alpha=0.6, label='Tumor',  color='salmon')
plt.axvline(0.5, color='k', linestyle='--', label='Threshold=0.5')
plt.xlabel('Predicted cancer probability')
plt.ylabel('Count')
plt.title('DNABERT-2 — Prediction Distribution')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dnabert2_pred_distribution.png'), dpi=150)
plt.show()

## 17 · Save final model & metrics to Drive

In [ ]:
import json

# Save final weights
final_dir = os.path.join(OUTPUT_DIR, 'final_checkpoint')
os.makedirs(final_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(final_dir, 'pytorch_model.pt'))
tokenizer.save_pretrained(final_dir)
model.encoder.config.save_pretrained(final_dir)
print(f'Final checkpoint saved to {final_dir}')

# Save history + metrics as JSON
metrics = {
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'best_val_auc': best_auc,
    'final_read_auc': final_auc,
    'cnn_baseline_auc': 0.615,
    'history': history,
}
if run_ids_test is not None:
    metrics['patient_auc'] = patient_auc

metrics_path = os.path.join(OUTPUT_DIR, 'dnabert2_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved to {metrics_path}')

print('\n' + '='*50)
print('SUMMARY')
print('='*50)
print(f'  CNN baseline read-level AUC : 0.615')
print(f'  DNABERT-2 read-level AUC    : {final_auc:.4f}')
if run_ids_test is not None:
    print(f'  DNABERT-2 patient-level AUC : {patient_auc:.4f}')
improvement = (final_auc - 0.615) * 100
print(f'  Improvement                 : {improvement:+.2f} pp')
print('='*50)